In [1]:
import numpy as np
import pandas as pd

# Display settings
pd.set_option("display.max_columns", None)

In [2]:
# ==============================================================================
# 1. CLIMATIC DATA ETL: TEMPERATURE, PRECIPITATION & SUNSHINE DURATION
# ==============================================================================

# File paths
ruta_temp = r"C:\Users\USUARIO\Documents\Datos_IDEAM\ideam_ciudad_bolivar_temperatura_mensual.csv"
ruta_precip = r"C:\Users\USUARIO\Documents\Datos_IDEAM\ideam_ciudad_bolivar_precipitacion_mensual.csv"
ruta_brillo = r"C:\Users\USUARIO\Documents\Datos_IDEAM\ideam_ciudad_bolivar_brillo_solar_mensual.csv"

# Load and filter Temperature
df_temp_raw = pd.read_csv(ruta_temp, sep=",", encoding="latin-1")
df_temp_raw["Fecha"] = pd.to_datetime(df_temp_raw["Fecha"])
df_temp = df_temp_raw[
    df_temp_raw["NombreEstacion"].str.contains("MANSA", case=False, na=False)
].copy()
df_temp = df_temp[["Fecha", "CodigoEstacion", "NombreEstacion", "Valor"]].rename(
    columns={"Valor": "temp_media_c"}
)

# Load and filter Precipitation
df_precip_raw = pd.read_csv(ruta_precip, sep=",", encoding="latin-1")
df_precip_raw["Fecha"] = pd.to_datetime(df_precip_raw["Fecha"])
df_precip = df_precip_raw[
    df_precip_raw["NombreEstacion"].str.contains("MANSA", case=False, na=False)
].copy()
df_precip = df_precip[["Fecha", "Valor"]].rename(
    columns={"Valor": "precipitacion_mm"}
)

# Load and filter Sunshine Hours
df_brillo_raw = pd.read_csv(ruta_brillo, sep=",", encoding="latin-1")
df_brillo_raw["Fecha"] = pd.to_datetime(df_brillo_raw["Fecha"])
df_brillo = df_brillo_raw[
    df_brillo_raw["NombreEstacion"].str.contains("MANSA", case=False, na=False)
].copy()
df_brillo = df_brillo[["Fecha", "Valor"]].rename(
    columns={"Valor": "brillo_solar_horas"}
)

# Merge climate series on Date
df_clima = df_temp.merge(df_precip, on="Fecha", how="inner").merge(
    df_brillo, on="Fecha", how="inner"
)

# Temporal helper dimensions
df_clima["anio"] = df_clima["Fecha"].dt.year
df_clima["mes"] = df_clima["Fecha"].dt.month

# Reorder columns
columnas_orden = [
    "Fecha",
    "anio",
    "mes",
    "CodigoEstacion",
    "NombreEstacion",
    "temp_media_c",
    "precipitacion_mm",
    "brillo_solar_horas",
]
df_clima = (
    df_clima[columnas_orden].sort_values("Fecha").reset_index(drop=True)
)

print(
    f"Base climática consolidada: {df_clima.shape[0]} meses ({df_clima['Fecha'].min().strftime('%Y-%m')} a {df_clima['Fecha'].max().strftime('%Y-%m')})"
)
print(f"Valores nulos detectados: {df_clima.isna().sum().sum()}")
df_clima.head()

Base climática consolidada: 124 meses (2015-01 a 2026-03)
Valores nulos detectados: 0


,Fecha,anio,mes,CodigoEstacion,NombreEstacion,temp_media_c,precipitacion_mm,brillo_solar_horas
0,2015-01-01,2015,1,11025010,LA MANSA [11025010],17.310573,135.4,175.6
1,2015-02-01,2015,2,11025010,LA MANSA [11025010],17.436111,60.3,120.8
2,2015-03-01,2015,3,11025010,LA MANSA [11025010],17.659537,152.9,154.0
3,2015-04-01,2015,4,11025010,LA MANSA [11025010],17.580342,232.5,129.7
4,2015-05-01,2015,5,11025010,LA MANSA [11025010],17.932906,391.7,164.5


In [3]:
# Save cleaned monthly climate dataset
ruta_salida_clima = r"C:\Users\USUARIO\Documents\Datos_IDEAM\clima_ciudad_bolivar_consolidado_2015_2024.csv"
df_clima.to_csv(ruta_salida_clima, index=False, encoding="utf-8")
print(f"Archivo exportado: {ruta_salida_clima}")

Archivo exportado: C:\Users\USUARIO\Documents\Datos_IDEAM\clima_ciudad_bolivar_consolidado_2015_2024.csv


In [5]:
# ==============================================================================
# 2. AGRICULTURAL PRODUCTION DATA (EVA / UPRA)
# ==============================================================================

produccion_cafe = r"C:\Users\USUARIO\Documents\Coffee_Climate_Agroanalysis\cafe_ciudad_bolivar_2015_2016.csv"

# Ingest handling UTF-8-BOM and Latin numeric formats (1.000,00)
df_cafe_raw = pd.read_csv(
    produccion_cafe, sep=",", encoding="utf-8-sig", thousands=".", decimal=","
)

# Standardize column naming convention
df_cafe = df_cafe_raw.rename(
    columns={
        "Año": "anio",
        "Departamento": "departamento",
        "Municipio": "municipio",
        "Producto": "producto",
        "Área Cosechada": "area_cosechada_ha",
        "Área Sembrada": "area_sembrada_ha",
        "Producción": "produccion_ton",
        "Rendimiento": "rendimiento_ton_ha",
        "Ciclo": "ciclo",
    }
)

df_cafe = df_cafe.sort_values("anio").reset_index(drop=True)
df_cafe.info()
df_cafe.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   anio                10 non-null     int64  
 1   departamento        10 non-null     str    
 2   municipio           10 non-null     str    
 3   producto            10 non-null     str    
 4   area_cosechada_ha   10 non-null     float64
 5   area_sembrada_ha    10 non-null     float64
 6   produccion_ton      10 non-null     float64
 7   rendimiento_ton_ha  10 non-null     float64
 8   ciclo               10 non-null     str    
dtypes: float64(4), int64(1), str(4)
memory usage: 852.0 bytes


,anio,departamento,municipio,producto,area_cosechada_ha,area_sembrada_ha,produccion_ton,rendimiento_ton_ha,ciclo
0,2015,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,8370.03,10245.20,9825.64,1.17,PERMANENTE
1,2016,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,8112.10,10128.70,9848.53,1.21,PERMANENTE
2,2017,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7416.11,10013.30,11248.80,1.52,PERMANENTE
3,2018,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7409.14,10071.70,10026.60,1.35,PERMANENTE
4,2019,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7441.00,9939.00,9903.00,1.33,PERMANENTE
5,2020,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7761.00,9902.00,9545.00,1.23,PERMANENTE
6,2021,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7365.18,9733.55,6775.97,0.92,PERMANENTE
7,2022,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7148.41,9405.83,7457.83,1.04,PERMANENTE
8,2023,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,7010.66,9048.01,7909.67,1.13,PERMANENTE
9,2024,ANTIOQUIA,CIUDAD BOLIVAR,CAFE,6586.65,8922.05,9221.31,1.40,PERMANENTE


In [7]:
# ==============================================================================
# 3. ANNUAL AGGREGATION & MULTIVARIATE INTEGRATION
# ==============================================================================

# Filtrar registros climáticos que coincidan con los años agrícolas (2015-2024)
clima_filtrado = df_clima[df_clima["anio"].between(2015, 2024)].copy()

# Agregación anual
clima_anual = (
    clima_filtrado.groupby("anio")
    .agg(
        temp_media_anual=("temp_media_c", "mean"),
        precipitacion_total_mm=("precipitacion_mm", "sum"),
        brillo_solar_total_horas=("brillo_solar_horas", "sum"),
        meses_registrados=("mes", "count"),
    )
    .reset_index()
)

# Cruce maestro por año
df_maestro = pd.merge(df_cafe, clima_anual, on="anio", how="inner")

columnas_analisis = [
    "anio",
    "rendimiento_ton_ha",
    "produccion_ton",
    "area_cosechada_ha",
    "temp_media_anual",
    "precipitacion_total_mm",
    "brillo_solar_total_horas",
    "meses_registrados",
]
df_maestro = df_maestro[columnas_analisis]

# Matriz de correlación frente al rendimiento
columnas_numericas = [
    "rendimiento_ton_ha",
    "temp_media_anual",
    "precipitacion_total_mm",
    "brillo_solar_total_horas",
]
matriz_corr = df_maestro[columnas_numericas].corr()

print("=== TABLA MAESTRA CONSOLIDADA ===")
display(df_maestro)

print("\n=== CORRELACIÓN DE PEARSON FRENTE AL RENDIMIENTO ===")
display(matriz_corr[["rendimiento_ton_ha"]].round(3))

=== TABLA MAESTRA CONSOLIDADA ===


,anio,rendimiento_ton_ha,produccion_ton,area_cosechada_ha,temp_media_anual,precipitacion_total_mm,brillo_solar_total_horas,meses_registrados
0,2015,1.17,9825.64,8370.03,17.803569,1524.9,1371.7,9
1,2016,1.21,9848.53,8112.10,17.691647,2333.0,1844.5,12
2,2017,1.52,11248.80,7416.11,17.289378,1876.8,1453.5,9
3,2018,1.35,10026.60,7409.14,17.098870,1898.7,1097.3,8
4,2019,1.33,9903.00,7441.00,17.517025,2121.2,1814.8,12
5,2020,1.23,9545.00,7761.00,17.286925,2485.0,1836.6,12
6,2021,0.92,6775.97,7365.18,17.033489,3099.2,1535.6,11
7,2022,1.04,7457.83,7148.41,16.747861,2842.7,1519.2,12
8,2023,1.13,7909.67,7010.66,17.553671,2414.2,1820.3,12
9,2024,1.40,9221.31,6586.65,17.856667,2428.3,2004.6,12



=== CORRELACIÓN DE PEARSON FRENTE AL RENDIMIENTO ===


,rendimiento_ton_ha
rendimiento_ton_ha,1.000
temp_media_anual,0.371
precipitacion_total_mm,-0.647
brillo_solar_total_horas,0.021


In [9]:
# ==============================================================================
# 4. EXPORT FINAL MASTER DATASET FOR TABLEAU
# ==============================================================================

cafe_tableau = r"C:\Users\USUARIO\Documents\Coffee_Climate_Agroanalysis\datos_cafe_clima_tableau.csv"
df_maestro.to_csv(cafe_tableau, index=False, encoding="utf-8-sig")

print(f"Dataset consolidado para Tableau exportado en: {cafe_tableau}")

Dataset consolidado para Tableau exportado en: C:\Users\USUARIO\Documents\Coffee_Climate_Agroanalysis\datos_cafe_clima_tableau.csv
